<a href="https://colab.research.google.com/github/leegunwoooo/inf/blob/main/day1_B_base_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 배송 지연 예측 — 1일차 B: 베이스 모델 구축
**담당: B (모델링)**  
**선행 조건:** A 담당자로부터 `processed_data.csv` 전달받은 후 실행

---
### 작업 목표
- LightGBM / XGBoost / Random Forest 기본 파이프라인 구축
- Stratified K-Fold 교차검증
- 성능 지표 확인 (Accuracy, F1, ROC-AUC, Confusion Matrix)
- 모델 저장 → 2일차 앙상블에서 활용

In [ ]:
 !pip install lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ─── 1. 라이브러리 임포트 ───
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac
matplotlib.rcParams['axes.unicode_minus'] = False
print('✅ 라이브러리 로드 완료')

✅ 라이브러리 로드 완료


In [ ]:
# ─── 2. 데이터 로드 (A 담당자 전달 파일) ───
DATA_PATH = '/content/synthetic_delivery_data.csv'
TARGET_COL = 'is_delayed'

df = pd.read_csv(DATA_PATH)
print(f'데이터 shape: {df.shape}')
print(f'타겟 분포:\n{df[TARGET_COL].value_counts()}')
df.head(3)

In [ ]:
# ─── 3. Feature / Target 분리 ───
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# 날짜·문자열 컬럼이 남아있으면 제거
X = X.select_dtypes(include=[np.number])

print(f'Feature 수: {X.shape[1]}')
print(f'클래스 비율 - 정상: {(y==0).sum()} / 지연: {(y==1).sum()}')

Feature 수: 24
클래스 비율 - 정상: 11138 / 지연: 3862


In [ ]:
# ─── 4. 모델 정의 ───
# 클래스 불균형 대비 scale_pos_weight 자동 계산
neg, pos = (y == 0).sum(), (y == 1).sum()
scale_w = round(neg / pos, 2)
print(f'scale_pos_weight: {scale_w}')

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=scale_w,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        scale_pos_weight=scale_w,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}
print('✅ 모델 정의 완료')

scale_pos_weight: 2.88
✅ 모델 정의 완료


In [ ]:
# ─── 5. Stratified K-Fold 교차검증 ───
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

cv_results = {}

for name, model in models.items():
    print(f'\n🔄 {name} 교차검증 중...')
    result = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        'Accuracy': result['test_accuracy'].mean(),
        'F1':       result['test_f1'].mean(),
        'ROC-AUC':  result['test_roc_auc'].mean(),
        'Acc_std':  result['test_accuracy'].std(),
        'F1_std':   result['test_f1'].std(),
        'AUC_std':  result['test_roc_auc'].std(),
    }
    print(f"  Accuracy : {cv_results[name]['Accuracy']:.4f} ± {cv_results[name]['Acc_std']:.4f}")
    print(f"  F1       : {cv_results[name]['F1']:.4f} ± {cv_results[name]['F1_std']:.4f}")
    print(f"  ROC-AUC  : {cv_results[name]['ROC-AUC']:.4f} ± {cv_results[name]['AUC_std']:.4f}")

print('\n✅ 교차검증 완료')


🔄 RandomForest 교차검증 중...
  Accuracy : 0.7185 ± 0.0062
  F1       : 0.4600 ± 0.0160
  ROC-AUC  : 0.6961 ± 0.0144

🔄 XGBoost 교차검증 중...
  Accuracy : 0.6962 ± 0.0069
  F1       : 0.4349 ± 0.0149
  ROC-AUC  : 0.6661 ± 0.0090

🔄 LightGBM 교차검증 중...
  Accuracy : 0.7141 ± 0.0064
  F1       : 0.4042 ± 0.0157
  ROC-AUC  : 0.6641 ± 0.0124

✅ 교차검증 완료


In [ ]:
# ─── 6. 성능 요약 테이블 출력 ───
summary = pd.DataFrame({
    name: {
        'Accuracy': f"{v['Accuracy']:.4f} ± {v['Acc_std']:.4f}",
        'F1':       f"{v['F1']:.4f} ± {v['F1_std']:.4f}",
        'ROC-AUC':  f"{v['ROC-AUC']:.4f} ± {v['AUC_std']:.4f}",
    }
    for name, v in cv_results.items()
}).T

print('\n📊 모델 성능 비교')
print(summary.to_string())


📊 모델 성능 비교
                     Accuracy               F1          ROC-AUC
RandomForest  0.7185 ± 0.0062  0.4600 ± 0.0160  0.6961 ± 0.0144
XGBoost       0.6962 ± 0.0069  0.4349 ± 0.0149  0.6661 ± 0.0090
LightGBM      0.7141 ± 0.0064  0.4042 ± 0.0157  0.6641 ± 0.0124


In [ ]:
# ─── 7. 모델 성능 시각화 ───
metrics = ['Accuracy', 'F1', 'ROC-AUC']
model_names = list(cv_results.keys())
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('베이스 모델 교차검증 성능 비교 (5-Fold)', fontsize=15, fontweight='bold')

metric_keys = [('Accuracy', 'Acc_std'), ('F1', 'F1_std'), ('ROC-AUC', 'AUC_std')]

for ax, (metric, std_key), title in zip(axes, metric_keys, metrics):
    vals = [cv_results[m][metric] for m in model_names]
    stds = [cv_results[m][std_key] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85,
                  yerr=stds, capsize=6, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontsize=13)
    ax.set_ylim(max(0, min(vals) - 0.1), 1.0)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('base_model_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 저장: base_model_cv_scores.png')

In [ ]:
# ─── 8. 전체 데이터로 최종 학습 + 모델 저장 ───
trained_models = {}

for name, model in models.items():
    print(f'📌 {name} 전체 데이터 학습 중...')
    model.fit(X, y)
    trained_models[name] = model
    joblib.dump(model, f'model_{name.lower()}.pkl')
    print(f'  → model_{name.lower()}.pkl 저장 완료')

print('\n✅ 모든 베이스 모델 저장 완료')
print('📂 저장된 파일:', [f'model_{n.lower()}.pkl' for n in models])

📌 RandomForest 전체 데이터 학습 중...
  → model_randomforest.pkl 저장 완료
📌 XGBoost 전체 데이터 학습 중...
  → model_xgboost.pkl 저장 완료
📌 LightGBM 전체 데이터 학습 중...
  → model_lightgbm.pkl 저장 완료

✅ 모든 베이스 모델 저장 완료
📂 저장된 파일: ['model_randomforest.pkl', 'model_xgboost.pkl', 'model_lightgbm.pkl']


In [ ]:
# ─── 9. Feature Importance (LightGBM 기준)
lgbm = trained_models['LightGBM']
feat_imp = pd.Series(lgbm.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
feat_imp.plot(kind='barh', color='#4C72B0', alpha=0.85)
plt.gca().invert_yaxis()
plt.title('Feature Importance Top 20 (LightGBM)', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 저장: feature_importance.png')

In [ ]:
# ─── 10. Confusion Matrix (LightGBM 마지막 Fold 기준) ───
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2,
                                             stratify=y, random_state=42)
lgbm_val = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    scale_pos_weight=scale_w, random_state=42, verbose=-1
)
lgbm_val.fit(X_tr, y_tr)
y_pred = lgbm_val.predict(X_val)

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['정상', '지연'], yticklabels=['정상', '지연'])
plt.title('Confusion Matrix — LightGBM (Validation)', fontsize=13)
plt.ylabel('실제'); plt.xlabel('예측')
plt.tight_layout()
plt.savefig('confusion_matrix_lgbm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Classification Report')
print(classification_report(y_val, y_pred, target_names=['정상', '지연']))